# Check 2: Is there usable satellite imagery of the flood?

**What we need:** two Sentinel-1 radar images of the same area:
- one **before** the flood (June 2026)
- one **during** the flood (about 18 July to 5 August 2026)

Both must come from the **same satellite track** (same relative orbit number and same pass direction). Otherwise the radar looks at the ground from a different angle and we see false changes.

**Before you start:** you need a Google Earth Engine account and a Cloud project registered for non-commercial use. Put your project ID in the cell below.

Note: Sentinel-1A stopped operating on 29 June 2026. Sentinel-1C and 1D now cover the area, about every 6 days, so there may be gaps.

In [ ]:
!pip install -q earthengine-api geemap

In [ ]:
# Get the project code (config, preprocessing, metrics) from GitHub.
# Change REPO_URL if your repository lives somewhere else.
import os, sys, subprocess
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
REPO = "/content/cutoff-flood"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from src import config
print("Project code loaded from", REPO)

In [ ]:
import ee
PROJECT_ID = "YOUR-PROJECT-ID"   # <-- your Earth Engine Cloud project ID
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine ready")

In [ ]:
import datetime
import pandas as pd

TODAY = datetime.date.today().isoformat()
RESULTS = f"{REPO}/results"
os.makedirs(RESULTS, exist_ok=True)

# Core three districts. To check coverage of all five affected districts,
# use config.ALL_AFFECTED_BBOX instead.
aoi = ee.Geometry.Rectangle(list(config.STUDY_BBOX))

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(aoi)
      .filterDate(*config.SEARCH_WINDOW)
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

aoi_area = aoi.area(1000)

def describe(img):
    cover = img.geometry().intersection(aoi, 1000).area(1000).divide(aoi_area)
    return ee.Feature(None, {
        "id": img.get("system:index"),
        "date": img.date().format("YYYY-MM-dd"),
        "time_utc": img.date().format("HH:mm"),
        "platform": img.get("platform_number"),
        "pass": img.get("orbitProperties_pass"),
        "orbit": img.get("relativeOrbitNumber_start"),
        "aoi_cover": cover,
    })

feats = s1.map(describe).getInfo()["features"]
df = pd.DataFrame([f["properties"] for f in feats]).sort_values("date")
df["aoi_cover"] = df["aoi_cover"].round(2)
df.to_csv(f"{RESULTS}/sentinel1_images_{TODAY}.csv", index=False)
print(len(df), "images found")
df

In [ ]:
# Find before/during pairs on the same track
pre = df[(df.date >= config.PRE_FLOOD_WINDOW[0]) & (df.date < config.PRE_FLOOD_WINDOW[1])]
post = df[(df.date >= config.FLOOD_WINDOW[0]) & (df.date <= config.FLOOD_WINDOW[1])]

pairs = pre.merge(post, on=["orbit", "pass"], suffixes=("_pre", "_post"))
pairs["min_cover"] = pairs[["aoi_cover_pre", "aoi_cover_post"]].min(axis=1)
pairs = pairs.sort_values(["min_cover", "date_post"], ascending=[False, True])

if pairs.empty:
    print("NO matching pair found. See the decision notes at the bottom.")
else:
    print(f"{len(pairs)} candidate pairs. Best ones first:")
pairs[["orbit", "pass", "date_pre", "date_post", "aoi_cover_pre", "aoi_cover_post", "id_pre", "id_post"]].head(10)

## Look at the best pair

The map shows the before and during images (VV band, lightly smoothed to reduce graininess). Open water is **dark**.

The red layer is a **rough** "new water" guess using a simple threshold (dark in July, not dark in June). It is only for this visual check, not our model. You should see red spreading over fields along the rivers. If the red is scattered randomly everywhere, the images may not show the flood.

In [ ]:
import geemap

PRE_ID = pairs.iloc[0]["id_pre"]     # or paste an id from the table
POST_ID = pairs.iloc[0]["id_post"]
print("Before:", PRE_ID)
print("During:", POST_ID)

def load(img_id):
    return (ee.Image(f"COPERNICUS/S1_GRD/{img_id}")
            .select(["VV", "VH"])
            .focalMedian(30, "circle", "meters"))

pre_img, post_img = load(PRE_ID), load(POST_ID)

ROUGH_WATER_DB = -16   # rough VV threshold for the quick look only
new_water = (post_img.select("VV").lt(ROUGH_WATER_DB)
             .And(pre_img.select("VV").gte(ROUGH_WATER_DB))).selfMask()

vis = {"bands": ["VV"], "min": -25, "max": 0}
Map = geemap.Map(center=[26.9, 94.6], zoom=10)
Map.addLayer(pre_img.clip(aoi), vis, "Before (June)")
Map.addLayer(post_img.clip(aoi), vis, "During (July)")
Map.addLayer(new_water.clip(aoi), {"palette": ["red"]}, "Rough new water")
Map

In [ ]:
# If the interactive map does not show, use these static previews instead.
from IPython.display import Image, display
params = {"region": aoi, "dimensions": 800, "min": -25, "max": 0, "bands": ["VV"]}
print("Before"); display(Image(url=pre_img.getThumbURL(params)))
print("During"); display(Image(url=post_img.getThumbURL(params)))

## Decision

Write the chosen image IDs and dates in `NOTES.md` (under "Check 2").

- **Pass:** a pair exists on the same track, covers most of the study area, and the flood is visible (new dark areas along rivers in July).
- **Partial:** a pair exists but covers only part of the area. Shrink the study area to what it covers.
- **Fail:** no usable July image. Try widening `FLOOD_WINDOW` in `src/config.py` by a few days, or test the backup area (Barpeta), where floods last longer and are easier to capture.